In [1]:
# loading API key
from dotenv import load_dotenv
import os
load_dotenv()

True

In [2]:
# checking it loads correctly
print(os.getenv("api_key") is not None)

True


In [11]:
api_key=os.getenv("api_key")

In [4]:
# getting csv
import pandas as pd
df_test = pd.read_csv("comp_llm_test.csv")

In [5]:
df_test.columns

Index(['competition_name', 'AI_level', 'AI_state', 'AI_age',
       'AI_primary_format_teams', 'AI_gender'],
      dtype='str')

In [6]:
len(df_test)

58

In [7]:
df_test.head()

,competition_name,AI_level,AI_state,AI_age,AI_primary_format_teams,AI_gender
0,AIFF Elite Youth League 2025-26,NaN,NaN,NaN,NaN,NaN
1,AIFF Sub Junior League 2025-26,NaN,NaN,NaN,NaN,NaN
2,AIFF Junior League 2025-26,NaN,NaN,NaN,NaN,NaN
3,Calcutta Football League - Premier Division 20...,NaN,NaN,NaN,NaN,NaN
4,Reliance Foundation Development League 2025-26,NaN,NaN,NaN,NaN,NaN


In [23]:
# converting all AI column types
ai_columns = [
    "AI_level",
    "AI_state",
    "AI_age",
    "AI_primary_format_teams",
    "AI_gender"
]

df_test[ai_columns] = df_test[ai_columns].astype("object")

In [12]:
# getting AI 
from openai import OpenAI

client = OpenAI(
    api_key=api_key,
    base_url="https://openrouter.ai/api/v1",
)

In [13]:
# testing connection
response = client.responses.create(
    model="gpt-5-mini",
    input="Reply with only the word: working"
)

print(response.output_text)

working


In [87]:
# defining prompt

classification_prompt = """
TASK AND PURPOSE

Classify each football competition consistently across five fields:

1. AI_primary_format_teams
2. AI_age
3. AI_level
4. AI_state
5. AI_gender

The purpose of this classification is to identify competitions relevant to an
analysis of player match appearances, particularly youth club leagues at the
national and state levels, while separating them from senior competitions,
cups, standalone tournaments and representative-team competitions.

The input will contain a football competition name in India. Classify that competition
using the rules below.

GENERAL INSTRUCTIONS

- Base every classification only on the competition name provided.
- Do not use web search or any external sources.
- Do not use prior knowledge about a competition that is not supported by its name.
- Classify a field only when the competition name provides sufficient evidence.
- If the competition name does not provide enough evidence for a field, assign "Unclear".
- Do not infer a category merely because it seems likely.
- Do not treat the word "championship" alone as proof that a competition is of a specific format.


When the competition name is not conclusive, consult, where available:

- AIFF fixtures and match-centre records;
- participating teams;
- venues;
- official competition pages and regulations;
- official AIFF material;
- official state- or district-association material;
- Wikipedia; and
- other reliable secondary sources.

Use the evidence gathered through this research to guide the classification.

--------------------------------------------------
1. AI_primary_format_teams
--------------------------------------------------

Determine the primary format in two steps.

STEP 1: IDENTIFY WHO PARTICIPATES

Ask:

What kind of entities is this competition designed for?

CLUB PARTICIPATION

Treat the competition as club football when the participating teams are
primarily:

- football clubs; or
- football academies.

The presence of the following does not change this conclusion, provided the
competition is still largely designed for clubs:

- institutional teams, such as police, railways, universities or services
  teams; and
- community sports associations or community-based clubs, such as Indrayani
  Sports Club or Sangam Sports Club.

OTHER PARTICIPATION

Treat the competition as involving other participants when:

- it involves representative teams of states or districts rather than clubs;
  or
- there is insufficient evidence that the competition is intended for clubs.

STEP 2: IDENTIFY THE COMPETITION FORMAT

Ask:

What is the primary mechanism through which the competition determines its
winner, relegation or progression?

CLUB LEAGUE

Assign "Club League" when the participating teams are clubs or academies and
one or more of the following applies:

- "League" appears in the competition's official name;
- the competition is officially described as a league;
- teams compete in one or more league stages in which they accumulate points
  over multiple scheduled matches; or
- league standings are the primary mechanism for determining the winner,
  relegation or progression.

A competition may still be classified as "Club League" if it includes:

- regional or zonal league stages;
- split-league formats;
- championship or relegation rounds; or
- a short playoff or knockout phase after the league stage, such as
  quarterfinals, semifinals or a final.

The presence of a knockout phase does not automatically make a competition
"Other". Determine whether the league structure and standings remain the
competition's primary mechanism.

OTHER

Assign "Other" when:

- it involves representative teams of states or districts rather than clubs;
- it is primarily an elimination tournament in which teams are eliminated
  after a loss rather than accumulating points across a league season;
- it begins with group or league stages, but those stages mainly function as
  qualification for a knockout tournament;
- there is insufficient evidence that it forms part of a club-league system;
  or
- it is officially organised as a cup, championship or standalone tournament
  rather than a league.

UNCLEAR

Assign "Unclear" when the primary format cannot be established from the 
competition name alone.after checking the available evidence.

EXAMPLES

- Elite Youth League -> Club League, despite having a knockout phase.
- Durand Cup -> Other, despite having a group stage.
- Santosh Trophy -> Other, because representative teams participate.
- A vaguely named "Youth Championship" with groups followed by knockouts ->
  Other or Unclear unless evidence shows that it belongs to a club-league
  system.

Allowed values for AI_primary_format_teams:

- Club League
- Other
- Unclear

--------------------------------------------------
2. AI_age
--------------------------------------------------

SENIOR

Assign "Senior" when:

- the competition is an open-age, professional or competitive platform
  intended for senior players rather than being governed primarily by a youth
  age restriction; and/or
- participation is based primarily on merit or playing ability rather than a
  maximum age.

A senior competition may include exceptional youth players who are capable of
competing physically and technically against adults. Their presence does not
make the competition a youth competition.

Competitions that emphasise transitional or emerging talent should still be
classified as "Senior" when selection is primarily based on ability rather
than participation being restricted to players below a specified age.

Examples:

- Poona District Football Association Second Division -> Senior
- GFA Second Division League -> Senior
- Indian Super League -> Senior

YOUTH

Assign "Youth" when participation is primarily governed by a maximum age and
the competition is intended for young players.

This includes:

- U-12, U-13, U-15, U-17, U-19, U-20 and U-21 competitions;
- Junior, Sub-Junior, Nursery or Youth leagues; and
- development leagues explicitly intended for young players, even when a
  limited number of over-age players are permitted.

Examples:

- Reliance Foundation Development League 2025-26 -> Youth
- GFA U-20 Division I League -> Youth

Terms such as Youth, Junior, Sub-Junior, Nursery, U-15 and U-17 are strong
evidence of a youth competition.

When the name is unclear, check official eligibility rules, competition
regulations or AIFF records.

UNCLEAR

Assign "Unclear" only when the age group cannot be established reliably after 
checking the available evidence.

Allowed values for AI_age:

- Youth
- Senior
- Unclear

--------------------------------------------------
3. AI_level
--------------------------------------------------

NATIONAL

Assign "National" when:

- the competition is organised at the national level by AIFF or AFC; or
- it involves teams from multiple Indian states as part of a national
  competition.

The mere presence of teams from different states does not necessarily make a
competition national when it is organised as a state, district or invitational
competition. Consider the organiser and the competition's formal scope.

STATE

Assign "State" when:

- it is organised by a state football association; or
- it functions as a state-wide competition.

DISTRICT

Assign "District" when:

- it is organised by a district football association; or
- it is conducted primarily within a district, city or city-level football
  structure.

Treat city association competitions as "District" for this classification.

UNCLEAR

Assign "Unclear" only when the competition level cannot be established from
the name or available evidence.

Allowed values for AI_level:

- National
- State
- District
- Unclear

--------------------------------------------------
4. AI_state
--------------------------------------------------

Record the Indian state or union territory most directly associated with the
competition.

Rules:

- For national competitions, assign "National".
- For state competitions, assign the state or union territory represented by
  the organising association or formal competition structure.
- For district or city competitions, assign the state or union territory in
  which that district or city is located.
- Do not assign a state solely because one match happened there when the
  competition itself belongs elsewhere.
- Use "Unclear" only when the associated state or union territory cannot be
  established reliably.
- Use full names rather than abbreviations.

Examples include (but are not limited to):

- AIFF → National
- GFA → Goa Football Association → Goa
- KSFA → Karnataka State Football Association → Karnataka
- WIFA → Western India Football Association → Maharashtra
- IFA → Indian Football Association → West Bengal
- DSA → Delhi Soccer Association → Delhi
- BDFA → Bangalore District Football Association → Karnataka
- PDFA / Poona District Football Association → Maharashtra

Use one of the following standard values:

Andaman and Nicobar Islands
Andhra Pradesh
Arunachal Pradesh
Assam
Bihar
Chandigarh
Chhattisgarh
Dadra and Nagar Haveli and Daman and Diu
Delhi
Goa
Gujarat
Haryana
Himachal Pradesh
Jammu and Kashmir
Jharkhand
Karnataka
Kerala
Ladakh
Lakshadweep
Madhya Pradesh
Maharashtra
Manipur
Meghalaya
Mizoram
Nagaland
Odisha
Puducherry
Punjab
Rajasthan
Sikkim
Tamil Nadu
Telangana
Tripura
Uttar Pradesh
Uttarakhand
West Bengal
National
Unclear

--------------------------------------------------
5. AI_gender
--------------------------------------------------

Classify gender using the competition name.

If the name explicitly contains Women's, Girls' or Ladies', assign Female.
Otherwise assign Male.

This dataset assumes competitions are men's/boys' competitions unless explicitly identified as women's competitions.
Allowed values for AI_gender:

- Male
- Female
- Unclear

--------------------------------------------------
OUTPUT REQUIREMENTS
--------------------------------------------------

Return only one valid JSON object. Do not include Markdown, introductory text,
headings or explanations outside the JSON.

Use these exact keys:

{
  "competition_name": "The competition name exactly as provided",
  "AI_level": "National, State, District or Unclear",
  "AI_state": "A standard state/union-territory value, National or Unclear",
  "AI_age": "Youth, Senior or Unclear",
  "AI_primary_format_teams": "Club League, Other or Unclear",
  "AI_gender": "Male, Female or Unknown",
}

Additional output rules:

- Copy competition_name exactly as provided. Do not correct, shorten or
  standardise it.
- Use only the permitted category values.
"""

In [61]:
# defining function for classification that loops on competition names and uses classificaiton prompt
import json

def classify_competition(competition_name):

    response = client.responses.create(
        model="gpt-5",
        input=f"""
{classification_prompt}

Competition name:
{competition_name}
"""
    )

    return json.loads(response.output_text)

In [62]:
# running the function on entire dataset 

for i, row in df_test.iterrows():

    try:
        result = classify_competition(row["competition_name"])

        df_test.loc[i, "AI_level"] = result["AI_level"]
        df_test.loc[i, "AI_state"] = result["AI_state"]
        df_test.loc[i, "AI_age"] = result["AI_age"]
        df_test.loc[i, "AI_primary_format_teams"] = result["AI_primary_format_teams"]
        df_test.loc[i, "AI_gender"] = result["AI_gender"]

        print(f"✅ Row {i+1}/{len(df_test)} done successfully")

    except Exception as e:
        print(f"❌ Row {i+1}/{len(df_test)} failed: {e}")

df_test.to_csv("competition_ai_test_completed.csv", index=False)

✅ Row 1/58 done successfully
✅ Row 2/58 done successfully
✅ Row 3/58 done successfully
✅ Row 4/58 done successfully
✅ Row 5/58 done successfully
✅ Row 6/58 done successfully
✅ Row 7/58 done successfully
✅ Row 8/58 done successfully
✅ Row 9/58 done successfully
✅ Row 10/58 done successfully
✅ Row 11/58 done successfully
✅ Row 12/58 done successfully
✅ Row 13/58 done successfully
✅ Row 14/58 done successfully
✅ Row 15/58 done successfully
✅ Row 16/58 done successfully
✅ Row 17/58 done successfully
✅ Row 18/58 done successfully
✅ Row 19/58 done successfully
✅ Row 20/58 done successfully
✅ Row 21/58 done successfully
✅ Row 22/58 done successfully
✅ Row 23/58 done successfully
✅ Row 24/58 done successfully
✅ Row 25/58 done successfully
✅ Row 26/58 done successfully
✅ Row 27/58 done successfully
✅ Row 28/58 done successfully
✅ Row 29/58 done successfully
✅ Row 30/58 done successfully
✅ Row 31/58 done successfully
✅ Row 32/58 done successfully
✅ Row 33/58 done successfully
✅ Row 34/58 done su

Accuracy achieved: 
handcoded_level: 69.0%
handcoded_state: 74.1%
handcoded_age: 98.3%
handcoded_primary_format_teams: 98.3%
handcoded_gender: 100.0%

## Running on entire dataset now

In [88]:
competitions = pd.read_csv("competition_percentages.csv")

In [89]:
competitions

,competition_name,player_rows,cum_pct
0,AIFF Elite Youth League 2025-26,31990,3.962263
1,AIFF Sub Junior League 2025-26,24043,6.940214
2,AIFF Junior League 2025-26,21229,9.569626
3,Calcutta Football League - Premier Division 20...,16073,11.560418
4,Reliance Foundation Development League 2025-26,15929,13.533375
...,...,...,...
440,KSFA Youth Premier League Sub Junior Tier-2 (U...,27,99.988357
441,KHELO INDIA ASMITA FOOTBALL LEAGUE UNDER-15,25,99.991454
442,THANE FOOTBALL PREMIER LEAGUE 2024 - 2025,24,99.994426
443,U-13 ASMITA CITY FOOTBALL LEAGUE 2025-2026 = R...,24,99.997399


In [90]:
ai_columns = [
    "AI_level",
    "AI_state",
    "AI_age",
    "AI_primary_format_teams",
    "AI_gender"
]

competitions[ai_columns] = pd.NA

In [91]:
for i, row in competitions.iterrows():

    try:
        result = classify_competition(row["competition_name"])

        competitions.loc[i, "AI_level"] = result["AI_level"]
        competitions.loc[i, "AI_state"] = result["AI_state"]
        competitions.loc[i, "AI_age"] = result["AI_age"]
        competitions.loc[i, "AI_primary_format_teams"] = result["AI_primary_format_teams"]
        competitions.loc[i, "AI_gender"] = result["AI_gender"]

        if (i + 1) % 25 == 0:
            competitions.to_csv("competition_ai_classified.csv", index=False)

        print(f"✅ Row {i+1}/{len(competitions)} done successfully")

    except Exception as e:
        print(f"❌ Row {i+1}/{len(competitions)} failed: {e}")

competitions.to_csv("competition_ai_classified.csv", index=False)

✅ Row 1/445 done successfully
✅ Row 2/445 done successfully
✅ Row 3/445 done successfully
✅ Row 4/445 done successfully
✅ Row 5/445 done successfully
✅ Row 6/445 done successfully
✅ Row 7/445 done successfully
✅ Row 8/445 done successfully
✅ Row 9/445 done successfully
✅ Row 10/445 done successfully
✅ Row 11/445 done successfully
✅ Row 12/445 done successfully
✅ Row 13/445 done successfully
✅ Row 14/445 done successfully
✅ Row 15/445 done successfully
✅ Row 16/445 done successfully
✅ Row 17/445 done successfully
✅ Row 18/445 done successfully
✅ Row 19/445 done successfully
✅ Row 20/445 done successfully
✅ Row 21/445 done successfully
✅ Row 22/445 done successfully
✅ Row 23/445 done successfully
✅ Row 24/445 done successfully
✅ Row 25/445 done successfully
✅ Row 26/445 done successfully
✅ Row 27/445 done successfully
✅ Row 28/445 done successfully
✅ Row 29/445 done successfully
✅ Row 30/445 done successfully
✅ Row 31/445 done successfully
✅ Row 32/445 done successfully
✅ Row 33/445 done